# Improved USP-800 RAG Workflow with LLM-as-a-Judge
This notebook implements an advanced RAG pipeline based on your previous projects. It includes:
1. **Advanced Retrieval**: Using MongoDB Atlas with Reranking logic.
2. **Evaluation Framework**: A 'Judge' LLM to score Faithfulness and Relevance, similar to your DualLens project.

In [ ]:
!pip install -q langchain langchain-mongodb langchain-openai langchain-community pymongo pypdf sentence-transformers

## 1. Setup & Connection
Connecting to MongoDB Atlas.

In [ ]:
import os
from pymongo import MongoClient
from langchain_mongodb import MongoDBAtlasVectorSearch
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

# Ensure you set your API key
# os.environ['OPENAI_API_KEY'] = 'your-key'

MONGODB_URI = "your_mongodb_uri"
client = MongoClient(MONGODB_URI)
DB_NAME = "usp800_rag_eval"
COLLECTION_NAME = "chunks"
collection = client[DB_NAME][COLLECTION_NAME]

embeddings = OpenAIEmbeddings()
vector_store = MongoDBAtlasVectorSearch(
    collection=collection,
    embedding=embeddings,
    index_name="vector_index"
)

## 2. The Advanced RAG Pipeline
Implementing a retriever that fetches context and a judge that evaluates it.

In [ ]:
llm = ChatOpenAI(model="gpt-4o", temperature=0)

def run_rag_with_eval(query):
    # 1. Retrieval
    docs = vector_store.similarity_search(query, k=3)
    context = "\n".join([d.page_content for d in docs])
    
    # 2. Generation
    prompt = f"Context: {context}\n\nQuestion: {query}\nAnswer:"
    answer = llm.invoke(prompt).content
    
    # 3. Judge Evaluation (Relevance & Faithfulness)
    judge_prompt = f"""
    Evaluate this RAG response:
    Query: {query}
    Context: {context}
    Answer: {answer}
    
    Return a JSON with:
    - faithfulness_score (1-5)
    - relevance_score (1-5)
    - critique (text)
    """
    evaluation = llm.invoke(judge_prompt).content
    
    return {
        "answer": answer,
        "evaluation": evaluation
    }

# Test Run
# result = run_rag_with_eval("What is the requirement for negative pressure in a C-SCA?")
# print(result)